In [25]:
from concurrent.futures import ThreadPoolExecutor
from collections import deque

import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
from scipy.ndimage import minimum_filter

import matplotlib.pyplot as plt

In [1]:
def get_windows(
    visual_tiff_url,
    window_width=520,
    window_height=520,
):
    """Yield tile IDs and Rasterio windows covering a visual TIFF."""
    if window_width <= 0 or window_height <= 0:
        raise ValueError("Window width and height must be positive.")

    with rasterio.open(visual_tiff_url) as src:
        image_width = src.width
        image_height = src.height

    tile_number = 0

    for row, y in enumerate(range(0, image_height, window_height)):
        for col, x in enumerate(range(0, image_width, window_width)):
            width = min(window_width, image_width - x)
            height = min(window_height, image_height - y)

            yield {
                "tile_id": f"r{row:05d}_c{col:05d}_n{tile_number:08d}",
                "row": row,
                "col": col,
                "window": Window(x, y, width, height),
            }

            tile_number += 1

In [3]:
def read_tile_with_halo(src, tile, halo=0):
    """Read one tile plus surrounding pixels, and return the RGB array.
    OUTPUT: rgb array with halo, initial tile slice without halo
    """
    window = tile["window"]

    # Expand the read area, keeping it inside the image bounds.
    x_start = max(0, int(window.col_off) - halo)
    y_start = max(0, int(window.row_off) - halo)
    x_end = min(src.width, int(window.col_off + window.width) + halo)
    y_end = min(src.height, int(window.row_off + window.height) + halo)

    read_window = Window(
        x_start,
        y_start,
        x_end - x_start,
        y_end - y_start,
    )

    # Rasterio returns (bands, height, width); rearrange to (height, width, bands).
    bands = src.read([1, 2, 3], window=read_window) # CHECK CHECK CHECK check the bands that the inital image have
    rgb = np.moveaxis(bands, 0, -1)

    # Find the original tile's position inside the larger, halo-inclusive array.
    row_start = int(window.row_off) - y_start
    col_start = int(window.col_off) - x_start

    tile_slice = (
        slice(row_start, row_start + int(window.height)), #rows
        slice(col_start, col_start + int(window.width)), #cols
    )

    return rgb, tile_slice

In [4]:
def get_red_blue(rgb):
    """Extract red and blue channels from an RGB image array.

    Args:
        rgb: NumPy array shaped (height, width, 3), ordered as
             red, green, blue.

    Returns:
        red, blue: 2D NumPy arrays, each shaped (height, width).
    """
    if rgb.ndim != 3 or rgb.shape[2] != 3:
        raise ValueError("Expected an RGB array shaped (height, width, 3).")

    red = rgb[:, :, 0]
    blue = rgb[:, :, 2]

    return red, blue

In [5]:
def estimate_clear_pixels(rgb, brightness_threshold=200, color_difference=30):
    """Select pixels that are not extremely bright and nearly white.

    Returns a Boolean mask: True means the pixel is included when
    fitting the clear-sky relationship.
    """
    if rgb.ndim != 3 or rgb.shape[2] != 3:
        raise ValueError("Expected an RGB array shaped (height, width, 3).")

    red = rgb[:, :, 0].astype(np.float32)
    green = rgb[:, :, 1].astype(np.float32)
    blue = rgb[:, :, 2].astype(np.float32)

    brightest_channel = np.maximum(np.maximum(red, green), blue)
    darkest_channel = np.minimum(np.minimum(red, green), blue)

    brightness = (red + green + blue) / 3
    nearly_white = (brightest_channel - darkest_channel) < color_difference
    bright = brightness > brightness_threshold

    # True means "use this pixel to fit the clear-sky line."
    return ~(bright & nearly_white)

In [20]:
def fit_clear_sky_line(red, blue, clear_mask):
    """Fit blue = slope * red + intercept using likely-clear pixels.

    Args:
        red: 2D array of red values.
        blue: 2D array of blue values, same shape as red.
        clear_mask: Boolean 2D array, same shape. True pixels are used
                    to fit the line.

    Returns:
        slope, intercept: Coefficients for the fitted line.
    """
    red = np.asarray(red, dtype=np.float64)
    blue = np.asarray(blue, dtype=np.float64)
    clear_mask = np.asarray(clear_mask, dtype=bool)

    if red.ndim != 2 or blue.shape != red.shape:
        raise ValueError("red and blue must be 2D arrays with the same shape.")
    if clear_mask.shape != red.shape:
        raise ValueError("clear_mask must have the same shape as red and blue.")

    # Use only selected pixels with finite channel values.
    valid = clear_mask & np.isfinite(red) & np.isfinite(blue)
    red_clear = red[valid]
    blue_clear = blue[valid]

    if red_clear.size < 2:
        raise ValueError("Need at least two valid clear pixels to fit a line.")

    slope, intercept = np.polyfit(red_clear, blue_clear, deg=1)
    return slope, intercept

In [7]:
def hot_score(red, blue, slope, intercept):
    """Calculate positive blue excess above the fitted clear-sky line.

    Args:
        red: 2D array of red values for all pixels.
        blue: 2D array of blue values, same shape as red.
        slope: Slope returned by fit_clear_sky_line.
        intercept: Intercept returned by fit_clear_sky_line.

    Returns:
        2D float array of HOT-style scores, same shape as red and blue.
        Larger positive scores indicate more blue than the line predicts.
    """
    red = np.asarray(red, dtype=np.float32)
    blue = np.asarray(blue, dtype=np.float32)

    if red.ndim != 2 or blue.shape != red.shape:
        raise ValueError("red and blue must be 2D arrays with the same shape.")

    predicted_blue = slope * red + intercept

    # Keep positive blue excess; set negative excess to zero.
    score = np.maximum(blue - predicted_blue, 0)
    score[~(np.isfinite(red) & np.isfinite(blue))] = np.nan

    return score

In [8]:
def make_hot_mask(score, threshold):
    """Convert HOT-style scores into a binary candidate cloud mask.

    Args:
        score: 2D array returned by hot_score.
        threshold: Score cutoff. Pixels above it are marked as candidates.

    Returns:
        2D uint8 array: 1 for candidate cloud, 0 otherwise.
    """
    score = np.asarray(score)

    if score.ndim != 2:
        raise ValueError("score must be a 2D array.")

    return ((score > threshold) & np.isfinite(score)).astype(np.uint8)

In [9]:
def resample_mask_to_label_grid(mask_path, label_path, output_path):
    """Resample a predicted mask to match the labeled GeoTIFF's grid."""
    with rasterio.open(mask_path) as mask_src, \
         rasterio.open(label_path) as label_src:

        # Use the label's dimensions and geospatial grid for the output.
        output_profile = label_src.profile.copy()
        output_profile.update(
            count=1,
            dtype="uint8",
            nodata=255,
            compress="deflate",
            photometric="MINISBLACK",
        )

        with rasterio.open(output_path, "w", **output_profile) as dst:
            reproject(
                source=rasterio.band(mask_src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=mask_src.transform,
                src_crs=mask_src.crs,
                dst_transform=label_src.transform,
                dst_crs=label_src.crs,
                src_nodata=mask_src.nodata,
                dst_nodata=255,
                resampling=Resampling.nearest,
            )

    return output_path

In [23]:
def fit_image_line(
    visual_tiff_url,
    window_width=520,
    window_height=520,
    brightness_threshold=200,
    color_difference=30,
    max_samples_per_tile=5000,
    random_seed=42,
):
    """Fit one clear-sky red-blue line for a full visual TIFF.

    Processes the image tile by tile, selects likely-clear pixels,
    and fits blue = slope * red + intercept.

    Returns:
        slope, intercept
    """
    rng = np.random.default_rng(random_seed)
    red_samples = []
    blue_samples = []

    with rasterio.open(visual_tiff_url) as src:
        for tile in get_windows(
            visual_tiff_url,
            window_width=window_width,
            window_height=window_height,
        ):
            rgb, _ = read_tile_with_halo(src, tile, halo=0)

            red, blue = get_red_blue(rgb)
            clear_mask = estimate_clear_pixels(
                rgb,
                brightness_threshold=brightness_threshold,
                color_difference=color_difference,
            )

            # Keep only likely-clear red and blue values.
            red_clear = red[clear_mask]
            blue_clear = blue[clear_mask]

            # Limit samples per tile to keep memory use manageable.
            if red_clear.size > max_samples_per_tile:
                chosen = rng.choice(
                    red_clear.size,
                    size=max_samples_per_tile,
                    replace=False,
                )
                red_clear = red_clear[chosen]
                blue_clear = blue_clear[chosen]

            red_samples.append(red_clear)
            blue_samples.append(blue_clear)

    if not red_samples or sum(samples.size for samples in red_samples) < 2:
        raise ValueError("Not enough likely-clear pixels to fit a line.")

    red_clear = np.concatenate(red_samples)
    blue_clear = np.concatenate(blue_samples)

    sample_mask = np.ones(red_clear.size, dtype=bool)

    return fit_clear_sky_line(
        red_clear.reshape(1, -1),
        blue_clear.reshape(1, -1),
        sample_mask.reshape(1, -1),
    )


In [11]:
def write_hot_mask_tiles(
    visual_tiff_url,
    output_path,
    slope,
    intercept,
    threshold,
    window_width=520,
    window_height=520,
):
    """Create and write a full-resolution HOT mask tile by tile.

    Args:
        visual_tiff_url: Path or URL to the RGB visual TIFF.
        output_path: Where to save the predicted mask TIFF.
        slope, intercept: Clear-sky line coefficients from fit_image_line.
        threshold: HOT score cutoff; scores above it are marked as cloud.
        window_width, window_height: Tile dimensions.

    Returns:
        output_path
    """
    with rasterio.open(visual_tiff_url) as src:
        if src.count < 3:
            raise ValueError("The visual TIFF must contain at least 3 bands.")

        # Make a single-band output TIFF on the visual image's grid.
        output_profile = src.profile.copy()
        output_profile.update(
            count=1,
            dtype="uint8",
            nodata=255,
            compress="deflate",
            photometric="MINISBLACK",
        )

        with rasterio.open(output_path, "w", **output_profile) as dst:
            for tile in get_windows(
                visual_tiff_url,
                window_width=window_width,
                window_height=window_height,
            ):
                rgb, _ = read_tile_with_halo(src, tile, halo=0)

                red, blue = get_red_blue(rgb)
                score = hot_score(red, blue, slope, intercept)
                tile_mask = make_hot_mask(score, threshold)

                # Write this tile into its matching location in the output.
                dst.write(tile_mask, 1, window=tile["window"])

    return output_path

In [21]:
import tempfile
from pathlib import Path

def generate_hot_mask(
    visual_tiff_url,
    label_tiff_url,
    output_path,
    threshold,
    window_width=520,
    window_height=520,
):
    """Fit the HOT line, create a temporary full-resolution mask,
    then resample it to the reference mask's grid.
    """
    slope, intercept = fit_image_line(
        visual_tiff_url,
        window_width=window_width,
        window_height=window_height,
    )

    with tempfile.TemporaryDirectory() as temp_dir:
        full_res_mask_path = str(Path(temp_dir) / "hot_mask_full_resolution.tif")

        write_hot_mask_tiles(
            visual_tiff_url=visual_tiff_url,
            output_path=full_res_mask_path,
            slope=slope,
            intercept=intercept,
            threshold=threshold,
            window_width=window_width,
            window_height=window_height,
        )

        resample_mask_to_label_grid(
            mask_path=full_res_mask_path,
            label_path=label_tiff_url,
            output_path=output_path,
        )

    return output_path

In [13]:
def get_visual_and_cloud_mask_tiff_urls(item_url):
    item_base = item_url.removesuffix(".json")

    visual_tiff_url = f"{item_base}-visual.tif"
    cloud_mask_tiff_url = f"{item_base}-clouds.tif"

    return visual_tiff_url, cloud_mask_tiff_url

In [36]:
item_url = "https://maxar-opendata.s3.amazonaws.com/events/tonga-volcano21/ard/01/300222120211/2022-01-18/10400100718A2000.json"
visual_tiff_url, label_tiff_url = get_visual_and_cloud_mask_tiff_urls(item_url)

thresholds = [10, 15, 20, 25, 30]

for threshold in thresholds:
    output_path = f"/content/output_cloud_mask{threshold}.tif"

    generate_hot_mask(
        visual_tiff_url,
        label_tiff_url,
        output_path,
        threshold=threshold,
    )

    display_masks_side_by_side(
        visual_tiff_url,
        output_path,
        label_tiff_url,
    )

Output hidden; open in https://colab.research.google.com to view.

Main Caveat
- incorrectly identifies water as cloud since it uses red, blue channels to evaluate clouds
- same things with bright surfaces like sand

In [26]:

def display_masks_side_by_side(visual_url, predicted_mask_path, label_mask_url):
    with rasterio.open(visual_url) as src:
        visual = np.moveaxis(src.read([1, 2, 3]), 0, -1)

    with rasterio.open(predicted_mask_path) as src:
        predicted_mask = src.read(1)

    with rasterio.open(label_mask_url) as src:
        label_mask = src.read(1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(visual)
    axes[0].set_title("Visual TIFF")

    axes[1].imshow(predicted_mask, cmap="gray", interpolation="nearest")
    axes[1].set_title("Predicted Cloud Mask")

    axes[2].imshow(label_mask, cmap="gray", interpolation="nearest")
    axes[2].set_title("Labeled Cloud Mask")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [34]:

# ---- Evaluation: compare the predicted mask against the provided mask -------------------------

def score_counts(tp, fp, fn, tn):
    d = lambda a, b: a / b if b else float("nan")
    return {
        "precision": d(tp, tp + fp),
        "recall": d(tp, tp + fn),
        "f1": d(2 * tp, 2 * tp + fp + fn),
        "iou": d(tp, tp + fp + fn),
        "accuracy": d(tp + tn, tp + fp + fn + tn),
    }
def evaluate_mask(predicted_mask_path, label_tiff_url, window_size=520):
    """Compare predicted cloud mask with the reference mask.

    Reference labels:
        1 = clear
        2 = cloud
        0 = NoData (ignored)
        3 = cloud shadow (ignored)

    Predicted mask:
        0 = clear
        1 = cloud
    """
    tp = fp = fn = tn = 0

    with rasterio.open(predicted_mask_path) as pred_src, \
         rasterio.open(label_tiff_url) as lab_src:

        if pred_src.width != lab_src.width or pred_src.height != lab_src.height:
            raise ValueError("Prediction and reference mask dimensions do not match.")

        for tile in get_windows(label_tiff_url, window_size, window_size):
            win = tile["window"]

            pred = pred_src.read(1, window=win)
            label = lab_src.read(1, window=win)

            # Evaluate only clear and cloud reference pixels.
            valid = (label == 1) | (label == 2)

            predicted_cloud = pred == 1
            reference_cloud = label == 2

            p = predicted_cloud[valid]
            g = reference_cloud[valid]

            tp += int((p & g).sum())
            fp += int((p & ~g).sum())
            fn += int((~p & g).sum())
            tn += int((~p & ~g).sum())

    out = score_counts(tp, fp, fn, tn)
    out.update(tp=tp, fp=fp, fn=fn, tn=tn)
    return out


In [35]:
metrics = evaluate_mask('/content/output_cloud_mask20.tif', label_tiff_url)
print("Overall evaluation metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

Overall evaluation metrics:
  precision: 0.4128221551670251
  recall: 0.18859086163768085
  f1: 0.2589052240994477
  iou: 0.14870254490628362
  accuracy: 0.3199861531044329
  tp: 508523
  fp: 723298
  fn: 2187912
  tn: 861371
